# Immunotherapy Response in Melanoma: Longitudinal Single-Cell Analysis

**Dataset**: Sade-Feldman et al., Cell 2018 (GSE120575)

This notebook demonstrates how to analyze longitudinal single-cell data from a clinical immunotherapy study, comparing immune dynamics between responders and non-responders to anti-PD-1 therapy.

## Background

Sade-Feldman et al. (Cell 2018) profiled tumor-infiltrating immune cells from melanoma patients **before and during** anti-PD-1 checkpoint inhibitor therapy. This landmark study identified transcriptional programs associated with clinical response.

### Study Design

**This is a prospective longitudinal study:**
- Patients received anti-PD-1 immunotherapy (pembrolizumab or nivolumab)
- Tumor biopsies collected **Pre-treatment** (baseline) and **Post-treatment** (on therapy)
- Response assessed by RECIST criteria: Complete/Partial Response vs Progressive Disease

**Key biological questions:**
- Do responders and non-responders have different baseline immune states?
- How does the immune microenvironment change with therapy?
- Are there response-specific trajectories (Difference-in-Differences)?

### Analysis Strategy

1. **Cross-sectional comparisons**: Responder vs Non-responder at each timepoint
2. **Within-arm longitudinal**: Pre→Post changes within each response group
3. **Difference-in-Differences (DiD)**: Do responders change differently than non-responders?

**Statistical considerations:**
- Participant-level aggregation to avoid pseudoreplication
- FDR correction for multiple testing
- Bootstrap inference for small sample sizes

## 1. Setup


In [1]:
# Imports - consolidated
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import pandas as pd
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.stats import mannwhitneyu, wilcoxon, ttest_ind
from statsmodels.stats.multitest import multipletests

import sctrial as st

# Configuration
MIN_GENES_FOR_SCORE = 5
MIN_PARTICIPANTS_FOR_COMPARISON = 3
FDR_ALPHA = 0.25
SEED = 42
RESPONSE_COL = "response_harmonized"

pd.options.mode.chained_assignment = None
print(f"sctrial version: {st.__version__ if hasattr(st, '__version__') else 'dev'}")


sctrial version: 0.2.1.dev1


## 2. Data Loading and Processing


In [2]:
from pathlib import Path
from io import StringIO
import gzip


def _resolve_dir_with_files(p: str, required_files) -> Path:
    """Find directory containing required files."""
    path = Path(p)
    if path.is_absolute():
        if all((path / f).exists() for f in required_files):
            return path
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / path
        if all((cand / f).exists() for f in required_files):
            return cand
    return path


def _looks_log1p(X, sample: int = 10000, seed: int = 0) -> bool:
    """Check if matrix appears to be log1p-transformed."""
    if X is None:
        return False
    if sp.issparse(X):
        data = X.data
    else:
        data = np.asarray(X).ravel()
    if data.size == 0:
        return False
    data = data[np.isfinite(data)]
    if data.size == 0:
        return False
    rng = np.random.default_rng(seed)
    if data.size > sample:
        data = rng.choice(data, size=sample, replace=False)
    # Log1p data: non-negative, bounded, non-integer
    return (data.min() >= 0) and (data.max() < 50) and (not np.allclose(data, np.round(data), atol=1e-3))


def load_sade_feldman(
    data_dir="data/sade_feldman",
    processed_name="sade_feldman_processed_v5.h5ad",
    max_cells_per_participant_visit=None,  # None = use all cells (no subsampling)
    seed=SEED,
    force_reprocess=False,
):
    """
    Load and preprocess Sade-Feldman melanoma immunotherapy dataset.
    
    Data source: GSE120575
    Paper: Sade-Feldman et al., Cell 2018
    
    Parameters
    ----------
    data_dir : str
        Directory containing raw data files
    processed_name : str
        Name for cached processed file
    max_cells_per_participant_visit : int or None
        Maximum cells per participant-visit combination. If None, uses all cells.
        Stratified sampling preserves longitudinal pairing.
    seed : int
        Random seed for subsampling
    force_reprocess : bool
        If True, reprocess even if cached file exists
    """
    data_dir = _resolve_dir_with_files(
        data_dir,
        [
            "GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz",
            "GSE120575_patient_ID_single_cells.txt.gz",
        ],
    )
    processed_path = data_dir.parent / "processed" / processed_name

    processing_params = {
        "version": "v5",
        "max_cells_per_participant_visit": max_cells_per_participant_visit,
        "seed": seed,
        "assay": "TPM",
    }

    if processed_path.exists() and not force_reprocess:
        adata = sc.read_h5ad(processed_path)
        prev = adata.uns.get("processing_params", {})
        if prev == processing_params:
            print(f"Loaded processed Sade-Feldman dataset: {adata.n_obs:,} cells, {adata.n_vars:,} genes")
            return adata
        else:
            print("Processed file parameters differ; reprocessing.")

    tpm_path = data_dir / "GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz"
    meta_path = data_dir / "GSE120575_patient_ID_single_cells.txt.gz"

    for p in [tpm_path, meta_path]:
        if not p.exists():
            raise FileNotFoundError(
                f"Missing file: {p}\n"
                f"Download from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE120575"
            )

    print("Processing raw data (this may take a minute)...")
    
    # Parse TPM matrix - has two header rows (sample IDs and time labels)
    with gzip.open(tpm_path, "rt") as f:
        header1 = f.readline().strip().split("\t")
        header2 = f.readline().strip().split("\t")
        if len(header1) != len(header2):
            raise ValueError("TPM file header rows have inconsistent lengths.")
        sample_ids = header1
        time_labels = header2
        data = f.read()

    df = pd.read_csv(StringIO(data), sep="\t", header=None)
    if df.iloc[:, -1].isna().all():
        df = df.iloc[:, :-1]

    genes = df.iloc[:, 0].astype(str).values
    mat = df.iloc[:, 1:]
    if mat.shape[1] != len(sample_ids):
        raise ValueError(f"TPM matrix columns ({mat.shape[1]}) != sample IDs ({len(sample_ids)}).")
    mat.columns = sample_ids

    # Parse metadata
    meta = pd.read_csv(meta_path, sep="\t", skiprows=19, encoding="latin1")
    meta = meta.rename(columns={
        "title": "sample_id",
        "characteristics: patinet ID (Pre=baseline; Post= on treatment)": "patient_raw",
        "characteristics: response": "response",
    })
    meta["sample_id"] = meta["sample_id"].astype(str)
    meta = meta.dropna(subset=["sample_id"]).copy()

    # Keep only real sample rows (format: Letter + digits + _P + digits + _M + digits)
    meta = meta[meta["sample_id"].str.match(r"^[A-Z]\d+_P\d+_M\d+")].copy()
    meta = meta[meta["response"].isin(["Responder", "Non-responder"])].copy()

    # Extract visit (Pre/Post) and participant ID
    meta["visit"] = meta["patient_raw"].astype(str).str.split("_").str[0]
    meta["participant_id"] = meta["patient_raw"].astype(str).str.extract(r"(P\d+)")[0]

    # Map time labels from header
    time_map = dict(zip(sample_ids, time_labels))
    meta["time_label"] = meta["sample_id"].map(time_map)

    meta = meta.set_index("sample_id")
    meta = meta.loc[[s for s in sample_ids if s in meta.index]].copy()

    # Create AnnData
    adata = sc.AnnData(X=mat.T.loc[meta.index].values.astype(np.float32))
    adata.obs = meta.copy()
    adata.var_names = genes

    # Note: Original dataset doesn't include cell type annotations
    adata.obs["cell_type"] = "Immune"  # Placeholder

    # Stratified subsampling (optional) to preserve longitudinal pairing
    if max_cells_per_participant_visit is not None:
        rng = np.random.default_rng(seed)
        keep_indices = []
        
        for (pid, visit), group in adata.obs.groupby(["participant_id", "visit"], observed=True):
            n_cells = len(group)
            if n_cells > max_cells_per_participant_visit:
                keep = rng.choice(group.index, size=max_cells_per_participant_visit, replace=False)
            else:
                keep = group.index.values
            keep_indices.extend(keep)
        
        adata = adata[keep_indices].copy()
        print(f"Stratified sampling: {adata.n_obs:,} cells (max {max_cells_per_participant_visit} per participant-visit)")
    else:
        print(f"Using full dataset: {adata.n_obs:,} cells (no subsampling)")

    # Store TPM and log1p(TPM) layers
    adata.layers["tpm"] = adata.X.copy()
    if _looks_log1p(adata.X):
        adata.layers["log1p_tpm"] = adata.X.copy()
    else:
        adata.layers["log1p_tpm"] = np.log1p(adata.X)

    adata.uns["processing_params"] = processing_params
    adata.uns["data_source"] = "GSE120575"
    adata.uns["paper"] = "Sade-Feldman et al., Cell 2018"

    # Save processed
    processed_path.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(processed_path)
    print(f"Saved processed file: {processed_path}")

    print(f"Loaded Sade-Feldman dataset: {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    return adata

### 2.1 Load processed AnnData

We harmonize response labels at the **participant level** (majority label) because a few participants have mixed response annotations across cells/samples. Mixed cases are reported below.


In [3]:
# Load data - using full dataset for reliable longitudinal analysis
adata = load_sade_feldman(max_cells_per_participant_visit=None)  # No subsampling

# Harmonize response labels at participant level
resp_counts = (
    adata.obs
    .groupby(["participant_id", "response"], observed=True)
    .size()
    .reset_index(name="n_cells")
)
resp_n = resp_counts.groupby("participant_id")["response"].nunique()
mixed_ids = resp_n[resp_n > 1].index.tolist()

dominant_response = (
    resp_counts
    .sort_values(["participant_id", "n_cells"], ascending=[True, False])
    .drop_duplicates("participant_id")
    .set_index("participant_id")["response"]
)

adata.obs[RESPONSE_COL] = adata.obs["participant_id"].map(dominant_response)
missing_resp = adata.obs[RESPONSE_COL].isna().sum()

print("")
print("=== Dataset Summary ===")
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")
print(f"Participants: {adata.obs['participant_id'].nunique()}")
print(f"Original response labels: {adata.obs['response'].unique().tolist()}")
print(f"Harmonized response labels: {adata.obs[RESPONSE_COL].unique().tolist()}")
print(f"Visits: {adata.obs['visit'].unique().tolist()}")

print("")
print("=== Response Label Consistency ===")
print(f"Participants with mixed response labels: {len(mixed_ids)}")
if mixed_ids:
    print("Using participant-level majority response for analyses.")
    display(
        resp_counts[resp_counts["participant_id"].isin(mixed_ids)]
        .sort_values(["participant_id", "n_cells"], ascending=[True, False])
    )
if missing_resp:
    print(f"WARNING: {missing_resp} cells are missing harmonized response labels.")

# Detailed pairing analysis
print("")
print("=== Longitudinal Pairing Analysis ===")
participant_visits = adata.obs.groupby("participant_id")["visit"].apply(set).reset_index()
participant_visits["has_Pre"] = participant_visits["visit"].apply(lambda x: "Pre" in x)
participant_visits["has_Post"] = participant_visits["visit"].apply(lambda x: "Post" in x)
participant_visits["is_paired"] = participant_visits["has_Pre"] & participant_visits["has_Post"]

# Add harmonized response info
participant_visits[RESPONSE_COL] = participant_visits["participant_id"].map(dominant_response)

print("")
print(f"Total participants: {len(participant_visits)}")
print(f"  Pre only: {(participant_visits['has_Pre'] & ~participant_visits['has_Post']).sum()}")
print(f"  Post only: {(~participant_visits['has_Pre'] & participant_visits['has_Post']).sum()}")
print(f"  Both (paired): {participant_visits['is_paired'].sum()}")

paired_by_resp = participant_visits[participant_visits["is_paired"]].groupby(RESPONSE_COL).size()
print("")
print("Paired participants by response (harmonized):")
for resp, count in paired_by_resp.items():
    print(f"  {resp}: {count}")

print("")
print(f"Obs columns: {sorted(adata.obs.columns.tolist())}")


FileNotFoundError: Missing file: data/sade_feldman/GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz
Download from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE120575

### 2.2 Quick exploratory summaries


In [4]:
# Sample size summary
print("=== Sample Sizes ===")
print("")

# Cells per group
cell_counts = adata.obs.groupby([RESPONSE_COL, "visit"], observed=True).size().unstack(fill_value=0)
print("Cells per Response x Visit:")
display(cell_counts)

# Participants per group
participant_counts = (
    adata.obs
    .groupby([RESPONSE_COL, "visit"], observed=True)["participant_id"]
    .nunique()
    .unstack(fill_value=0)
)
print("")
print("Participants per Response x Visit:")
display(participant_counts)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Cells by response
adata.obs[RESPONSE_COL].value_counts().plot(
    kind="bar", ax=axes[0], color=["forestgreen", "coral"]
)
axes[0].set_title("Cells by Response (harmonized)")
axes[0].set_ylabel("Number of cells")

# Cells by visit
adata.obs["visit"].value_counts().plot(
    kind="bar", ax=axes[1], color=["steelblue", "orange"]
)
axes[1].set_title("Cells by Visit")

# Participants per group
participant_counts.T.plot(kind="bar", ax=axes[2])
axes[2].set_title("Participants per Group")
axes[2].set_ylabel("Number of participants")
axes[2].legend(title="Response (harmonized)")

plt.tight_layout()
plt.show()


=== Sample Sizes ===



NameError: name 'adata' is not defined

## 3. Trial Design and Timepoint Strategy


In [5]:
# Define study design
visit_col = "visit"
adata.obs[visit_col] = adata.obs[visit_col].astype(str)
visits = [v for v in ["Pre", "Post"] if v in adata.obs[visit_col].unique()]
print(f"Available visits: {visits}")

# Participant-level response mapping (harmonized)
participant_response = adata.obs.groupby("participant_id")[RESPONSE_COL].first()

# Check longitudinal pairing (participant level)
participant_summary = (
    adata.obs.groupby("participant_id")[visit_col].apply(set).reset_index()
)
participant_summary["has_Pre"] = participant_summary[visit_col].apply(lambda x: "Pre" in x)
participant_summary["has_Post"] = participant_summary[visit_col].apply(lambda x: "Post" in x)
participant_summary["is_paired"] = participant_summary["has_Pre"] & participant_summary["has_Post"]
participant_summary[RESPONSE_COL] = participant_summary["participant_id"].map(participant_response)

paired_ids = set(participant_summary.loc[participant_summary["is_paired"], "participant_id"])
n_paired = len(paired_ids)

# Paired participants by response (harmonized)
paired_by_response = (
    participant_summary[participant_summary["is_paired"]]
    .groupby(RESPONSE_COL)
    .size()
    .to_dict()
)

paired_ids_by_response = {
    arm: set(
        participant_summary[
            (participant_summary["is_paired"]) & (participant_summary[RESPONSE_COL] == arm)
        ]["participant_id"]
    )
    for arm in ["Responder", "Non-responder"]
}

print("")
print("Longitudinal pairing:")
print(f"  Total paired participants (Pre + Post): {n_paired}")
print(f"  Paired Responders: {paired_by_response.get('Responder', 0)}")
print(f"  Paired Non-responders: {paired_by_response.get('Non-responder', 0)}")

# Check if DiD analysis is feasible
MIN_PAIRED_PER_ARM = 3
can_do_did = (
    paired_by_response.get("Responder", 0) >= MIN_PAIRED_PER_ARM and
    paired_by_response.get("Non-responder", 0) >= MIN_PAIRED_PER_ARM
)

if can_do_did:
    print("")
    print(f"  DiD analysis is feasible (>={MIN_PAIRED_PER_ARM} paired per arm)")
else:
    print("")
    print(f"  WARNING: DiD analysis may be underpowered (<{MIN_PAIRED_PER_ARM} paired in one arm)")

# Define trial design for sctrial
# Note: "Responder" is the group of interest (like "treated" in a trial context)
design = st.TrialDesign(
    participant_col="participant_id",
    visit_col=visit_col,
    arm_col=RESPONSE_COL,
    arm_treated="Responder",      # Group of primary interest
    arm_control="Non-responder",  # Comparison group
    celltype_col="cell_type",
)

print("")
print("Design configured:")
print(f"  Participant: {design.participant_col}")
print(f"  Visit: {design.visit_col}")
print(f"  Comparison: {design.arm_treated} vs {design.arm_control}")


NameError: name 'adata' is not defined

## 4. Immune Signatures

We define gene signatures relevant to immunotherapy response, based on the original Sade-Feldman paper and broader immuno-oncology literature:

- **Cytotoxicity**: Effector function of CD8 T cells and NK cells - associated with anti-tumor immunity
- **Exhaustion**: Dysfunctional T cell state - elevated exhaustion may predict poor response
- **IFN Response**: Interferon-gamma signaling - can indicate immune activation
- **Memory**: Memory T cell markers - may predict durable responses
- **Activation**: T cell activation markers - early response indicator

In [6]:
available_genes = set(adata.var_names)

# Define immunotherapy-relevant gene signatures
# Based on Sade-Feldman et al. and broader immuno-oncology literature
gene_signatures = {
    "Cytotoxicity": [
        "GZMB", "GZMA", "GZMH", "GZMK", "PRF1", "GNLY", 
        "IFNG", "NKG7", "KLRD1", "KLRB1", "FASLG"
    ],
    "Exhaustion": [
        "PDCD1", "LAG3", "HAVCR2", "TIGIT", "CTLA4", 
        "TOX", "ENTPD1", "CXCL13", "EOMES"
    ],
    "IFN_Response": [
        "ISG15", "IFI6", "IFIT1", "IFIT2", "IFIT3", 
        "MX1", "MX2", "STAT1", "OAS1", "IRF7"
    ],
    "Memory": [
        "IL7R", "TCF7", "LEF1", "CCR7", "SELL", "CD27", "CD28"
    ],
    "Activation": [
        "CD69", "CD38", "HLA-DRA", "ICOS", "CD44", "IL2RA"
    ],
}

# Filter to available genes and report coverage
print("Gene signature coverage:")
print("-" * 50)
filtered_signatures = {}
for name, genes in gene_signatures.items():
    found = [g for g in genes if g in available_genes]
    pct = len(found) / len(genes) * 100
    status = "OK" if len(found) >= MIN_GENES_FOR_SCORE else "SKIP"
    print(f"{name}: {len(found)}/{len(genes)} genes ({pct:.0f}%) [{status}]")
    if len(found) >= MIN_GENES_FOR_SCORE:
        filtered_signatures[name] = found

# Score gene sets using z-mean method
# zmean: z-score each gene across cells, then average z-scores
# This accounts for different expression scales across genes
if filtered_signatures:
    adata = st.score_gene_sets(
        adata, 
        filtered_signatures, 
        layer="log1p_tpm", 
        method="zmean",  # Better than "mean" for combining genes
        prefix="sig_"
    )
    print(f"\nScored {len(filtered_signatures)} signatures using zmean method")
else:
    print(f"\nNo gene sets passed threshold (min_genes={MIN_GENES_FOR_SCORE})")

# Get signature columns
signature_cols = [c for c in adata.obs.columns if c.startswith("sig_")]
print(f"Signature scores: {signature_cols}")

# Filter out features with ~zero variance
features_use = []
if signature_cols:
    df_feat = adata.obs[[design.participant_col, design.visit_col] + signature_cols].copy()
    df_feat = df_feat[df_feat[design.visit_col].isin(visits)]
    df_agg = df_feat.groupby([design.participant_col, design.visit_col], observed=True)[signature_cols].mean().reset_index()
    
    for f in signature_cols:
        if df_agg[f].std(ddof=1) > 1e-6:
            features_use.append(f)
        else:
            print(f"  Dropping {f}: near-zero variance")

print(f"\nFeatures for analysis: {features_use}")

NameError: name 'adata' is not defined

## 5. Cross-Sectional Comparisons by Timepoint

Compare Responders vs Non-responders at each visit (Pre and Post).

**Interpretation:**
- Positive beta = higher in Responders
- Negative beta = higher in Non-responders

In [7]:
print("=" * 60)
print("CROSS-SECTIONAL ANALYSIS: Responder vs Non-responder")
print("=" * 60)

cross_sectional_results = []

if features_use:
    for v in visits:
        # Check sample sizes
        sub = adata[adata.obs[design.visit_col] == v]
        n_per_arm = sub.obs.groupby(design.arm_col)[design.participant_col].nunique().to_dict()
        n_resp = n_per_arm.get(design.arm_treated, 0)
        n_nonresp = n_per_arm.get(design.arm_control, 0)
        
        print("")
        print(f"{v}: Responders={n_resp}, Non-responders={n_nonresp} participants")
        
        if n_resp < MIN_PARTICIPANTS_FOR_COMPARISON or n_nonresp < MIN_PARTICIPANTS_FOR_COMPARISON:
            print(f"  Skipping: insufficient participants (need >={MIN_PARTICIPANTS_FOR_COMPARISON} per arm)")
            continue
        
        # Use sctrial's built-in function
        res = st.between_arm_comparison(
            adata,
            visit=v,
            features=features_use,
            design=design,
            aggregate="participant_visit",
            standardize=True,
            method="ols",
        )
        
        if not res.empty:
            res["visit"] = v
            cross_sectional_results.append(res)
            
            # Display
            display_cols = ["feature", "beta_arm", "p_arm", "FDR_arm", "n_units"]
            print("")
            print(f"Results at {v}:")
            display(res[display_cols].round(4))
            
            # Highlight significant
            sig = res[res["FDR_arm"] < FDR_ALPHA]
            if not sig.empty:
                print(f"  Significant (FDR<{FDR_ALPHA}): {sig['feature'].tolist()}")
else:
    print("No features available for cross-sectional comparison.")

# Combine all results
if cross_sectional_results:
    all_cross = pd.concat(cross_sectional_results, ignore_index=True)
else:
    all_cross = pd.DataFrame()


CROSS-SECTIONAL ANALYSIS: Responder vs Non-responder


NameError: name 'features_use' is not defined

## 6. Within-Arm Longitudinal Comparisons

Test whether signatures change from Pre to Post **within each response group**.

This answers: "Do Responders show changes over time?" and "Do Non-responders show changes?"

In [8]:
print("=" * 60)
print("WITHIN-ARM LONGITUDINAL ANALYSIS: Pre → Post changes")
print("=" * 60)
print("")
print("Using paired Wilcoxon signed-rank test (appropriate for small samples)")

within_arm_results = []

if features_use and len(visits) == 2:
    for arm in [design.arm_treated, design.arm_control]:
        # Check pairing for this arm
        paired_ids_arm = paired_ids_by_response.get(arm, set())
        n_paired_arm = len(paired_ids_arm)
        print("")
        print(f"{arm}: {n_paired_arm} paired participants")
        
        if n_paired_arm < MIN_PARTICIPANTS_FOR_COMPARISON:
            print(f"  Skipping: need >= {MIN_PARTICIPANTS_FOR_COMPARISON} paired participants")
            continue
        
        # Subset to this arm and paired participants
        ad_arm = adata[
            (adata.obs[design.arm_col] == arm) &
            (adata.obs[design.participant_col].isin(paired_ids_arm))
        ].copy()
        ad_arm = ad_arm[ad_arm.obs[design.visit_col].isin(visits)].copy()
        
        # Aggregate to participant-visit level
        df_agg = (
            ad_arm.obs
            .groupby([design.participant_col, design.visit_col], observed=True)[features_use]
            .mean()
            .reset_index()
        )
        
        # Pivot to wide format for paired testing
        arm_rows = []
        for feat in features_use:
            wide = df_agg.pivot(
                index=design.participant_col, 
                columns=design.visit_col, 
                values=feat
            )
            
            # Keep only paired (have both Pre and Post)
            if visits[0] not in wide.columns or visits[1] not in wide.columns:
                continue
            wide = wide.dropna()
            
            if len(wide) < 3:
                arm_rows.append({
                    "feature": feat,
                    "n_paired": len(wide),
                    "mean_Pre": np.nan,
                    "mean_Post": np.nan,
                    "mean_delta": np.nan,
                    "p_time": np.nan,
                })
                continue
            
            pre_vals = wide[visits[0]].values
            post_vals = wide[visits[1]].values
            delta = post_vals - pre_vals
            
            # Wilcoxon signed-rank test (paired, non-parametric)
            try:
                stat, p_val = wilcoxon(delta)
            except Exception:
                p_val = np.nan
            
            arm_rows.append({
                "feature": feat,
                "n_paired": len(wide),
                "mean_Pre": float(pre_vals.mean()),
                "mean_Post": float(post_vals.mean()),
                "mean_delta": float(delta.mean()),
                "p_time": float(p_val),
            })
        
        if arm_rows:
            df_arm = pd.DataFrame(arm_rows)
            
            # FDR correction
            mask = df_arm["p_time"].notna()
            df_arm["FDR_time"] = np.nan
            if mask.sum() > 0:
                df_arm.loc[mask, "FDR_time"] = multipletests(
                    df_arm.loc[mask, "p_time"], method="fdr_bh"
                )[1]
            
            df_arm["arm"] = arm
            within_arm_results.append(df_arm)
            
            # Display
            print("")
            print(f"Pre→Post changes in {arm}:")
            display_cols = ["feature", "n_paired", "mean_delta", "p_time", "FDR_time"]
            display(df_arm[display_cols].round(4))
            
            # Highlight significant
            sig = df_arm[(df_arm["FDR_time"].notna()) & (df_arm["FDR_time"] < FDR_ALPHA)]
            if not sig.empty:
                for _, row in sig.iterrows():
                    direction = "↑" if row["mean_delta"] > 0 else "↓"
                    print(f"  {row['feature']}: {direction} (delta={row['mean_delta']:.3f}, FDR={row['FDR_time']:.3f})")
else:
    print("Insufficient visits or features for within-arm comparison.")

# Combine results
if within_arm_results:
    all_within = pd.concat(within_arm_results, ignore_index=True)
else:
    all_within = pd.DataFrame()


WITHIN-ARM LONGITUDINAL ANALYSIS: Pre → Post changes

Using paired Wilcoxon signed-rank test (appropriate for small samples)


NameError: name 'features_use' is not defined

## 7. Difference-in-Differences (DiD) Analysis

The key question: **Do Responders change differently than Non-responders?**

DiD tests whether the Pre→Post change differs between response groups:
- `beta_DiD > 0`: Responders increase more (or decrease less) than Non-responders
- `beta_DiD < 0`: Non-responders increase more (or decrease less) than Responders

This is the interaction effect: `(Post - Pre)_Responder - (Post - Pre)_Non-responder`

In [9]:
print("=" * 60)
print("DIFFERENCE-IN-DIFFERENCES ANALYSIS")
print("=" * 60)
print("")
print("Using participant-level deltas with Mann-Whitney U test")
print("(More robust for small samples than fixed-effects models)")

did_results = None

if features_use and len(visits) == 2:
    total_paired = paired_by_response.get('Responder', 0) + paired_by_response.get('Non-responder', 0)
    
    print("")
    print("Paired participants available:")
    print(f"  Responders: {paired_by_response.get('Responder', 0)}")
    print(f"  Non-responders: {paired_by_response.get('Non-responder', 0)}")
    print(f"  Total: {total_paired}")
    
    if paired_by_response.get('Responder', 0) < 3 or paired_by_response.get('Non-responder', 0) < 3:
        print("")
        print("⚠️  DiD requires >= 3 paired participants per arm.")
        print("   Insufficient pairing for reliable DiD analysis.")
    else:
        print("")
        print("Calculating DiD...")
        
        # Aggregate to participant-visit level (paired participants only)
        df_agg = (
            adata.obs[adata.obs[design.participant_col].isin(paired_ids)]
            .groupby([design.participant_col, design.visit_col], observed=True)[features_use]
            .mean()
            .reset_index()
        )
        
        did_rows = []
        for feat in features_use:
            # Pivot to get Pre/Post for each participant
            wide = df_agg.pivot_table(
                index=design.participant_col,
                columns=design.visit_col,
                values=feat,
                aggfunc="mean"
            )
            
            if visits[0] not in wide.columns or visits[1] not in wide.columns:
                continue
            
            # Calculate deltas
            wide["delta"] = wide[visits[1]] - wide[visits[0]]
            wide = wide.dropna(subset=["delta"])
            
            # Get arm for each participant (harmonized)
            wide["arm"] = wide.index.map(participant_response)
            
            # Split by arm
            delta_resp = wide[wide["arm"] == design.arm_treated]["delta"].values
            delta_nonresp = wide[wide["arm"] == design.arm_control]["delta"].values
            
            if len(delta_resp) < 3 or len(delta_nonresp) < 3:
                did_rows.append({
                    "feature": feat,
                    "n_resp": len(delta_resp),
                    "n_nonresp": len(delta_nonresp),
                    "delta_resp": np.nan,
                    "delta_nonresp": np.nan,
                    "beta_DiD": np.nan,
                    "p_DiD": np.nan,
                })
                continue
            
            # DiD estimate: (Post-Pre)_Resp - (Post-Pre)_NonResp
            beta_did = delta_resp.mean() - delta_nonresp.mean()
            
            # Mann-Whitney U test on the deltas
            try:
                stat, p_val = mannwhitneyu(delta_resp, delta_nonresp, alternative="two-sided")
            except Exception:
                p_val = np.nan
            
            did_rows.append({
                "feature": feat,
                "n_resp": len(delta_resp),
                "n_nonresp": len(delta_nonresp),
                "delta_resp": float(delta_resp.mean()),
                "delta_nonresp": float(delta_nonresp.mean()),
                "beta_DiD": float(beta_did),
                "p_DiD": float(p_val),
            })
        
        if did_rows:
            did_results = pd.DataFrame(did_rows)
            
            # FDR correction
            mask = did_results["p_DiD"].notna()
            did_results["FDR_DiD"] = np.nan
            if mask.sum() > 0:
                did_results.loc[mask, "FDR_DiD"] = multipletests(
                    did_results.loc[mask, "p_DiD"], method="fdr_bh"
                )[1]
            
            # Sort by p-value
            did_results = did_results.sort_values("p_DiD")
            
            print("")
            print("DiD Results:")
            display_cols = ["feature", "n_resp", "n_nonresp", "delta_resp", "delta_nonresp", "beta_DiD", "p_DiD", "FDR_DiD"]
            display(did_results[display_cols].round(4))
            
            # Interpretation
            print("")
            print("Interpretation:")
            print("  beta_DiD > 0: Responders increase MORE (or decrease less) than Non-responders")
            print("  beta_DiD < 0: Non-responders increase MORE (or decrease less) than Responders")
            
            # Highlight significant
            sig = did_results[(did_results["FDR_DiD"].notna()) & (did_results["FDR_DiD"] < FDR_ALPHA)]
            if not sig.empty:
                print("")
                print(f"Significant DiD effects (FDR < {FDR_ALPHA}):")
                for _, row in sig.iterrows():
                    direction = "Responders ↑ more" if row["beta_DiD"] > 0 else "Non-responders ↑ more"
                    print(f"  {row['feature']}: {direction} (β={row['beta_DiD']:.3f})")
            else:
                print("")
                print(f"No signatures showed significant differential change (FDR < {FDR_ALPHA})")
else:
    print("DiD analysis skipped: insufficient visits or features")


DIFFERENCE-IN-DIFFERENCES ANALYSIS

Using participant-level deltas with Mann-Whitney U test
(More robust for small samples than fixed-effects models)


NameError: name 'features_use' is not defined

## 8. Visualizations

### 8.1 Signature Distributions by Response and Visit

In [10]:
# Visualize signature distributions
if features_use:
    n_features = len(features_use)
    n_cols = min(3, n_features)
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    if n_features == 1:
        axes = np.array([[axes]])
    axes = axes.flatten() if n_features > 1 else [axes]
    
    for i, feat in enumerate(features_use):
        ax = axes[i]
        
        # Aggregate to participant level for visualization
        df_plot = (
            adata.obs
            .groupby(["participant_id", RESPONSE_COL, "visit"], observed=True)[feat]
            .mean()
            .reset_index()
        )
        
        sns.boxplot(
            data=df_plot, x="visit", y=feat, hue=RESPONSE_COL,
            palette={"Responder": "forestgreen", "Non-responder": "coral"},
            ax=ax, order=["Pre", "Post"]
        )
        ax.set_title(feat.replace("sig_", ""))
        ax.set_xlabel("Visit")
        ax.set_ylabel("Score (z-mean)")
        if i > 0:
            ax.get_legend().remove()
    
    # Hide unused axes
    for j in range(i+1, len(axes)):
        axes[j].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("No features to visualize.")


NameError: name 'features_use' is not defined

### 8.2 Trial Interaction Plot

Shows mean trajectories for each response group from Pre to Post.

In [11]:
# Interaction plots for top features
if features_use and len(visits) == 2:
    n_plots = min(len(features_use), 4)
    fig, axes = plt.subplots(1, n_plots, figsize=(4*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    
    for i, feat in enumerate(features_use[:n_plots]):
        ax = axes[i]
        try:
            st.plot_trial_interaction(
                adata, feat, design=design, visits=tuple(visits), ax=ax
            )
            ax.set_title(feat.replace("sig_", ""))
        except Exception as e:
            ax.text(0.5, 0.5, f"Could not plot: {feat}", 
                   ha="center", va="center", transform=ax.transAxes)
            ax.set_title(feat.replace("sig_", ""))
    
    plt.tight_layout()
    plt.show()
else:
    print("Skipping interaction plots: insufficient visits or features.")

# DiD Bar Plot (since we don't have SE for forest plot)
if did_results is not None and not did_results.empty:
    valid_did = did_results[did_results["beta_DiD"].notna()]
    if not valid_did.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        
        colors = ["forestgreen" if b > 0 else "coral" for b in valid_did["beta_DiD"]]
        bars = ax.barh(valid_did["feature"], valid_did["beta_DiD"], color=colors)
        ax.axvline(0, color="black", linewidth=0.5)
        ax.set_xlabel("DiD Effect (β)")
        ax.set_title("Difference-in-Differences: Responder vs Non-responder (Green = Responders ↑ more, Red = Non-responders ↑ more)")
        
        # Add significance markers
        for i, (_, row) in enumerate(valid_did.iterrows()):
            if pd.notna(row["FDR_DiD"]) and row["FDR_DiD"] < FDR_ALPHA:
                ax.text(row["beta_DiD"], i, " *", va="center", fontsize=14, fontweight="bold")
        
        plt.tight_layout()
        plt.show()
    else:
        print("Skipping DiD plot: no valid results.")
else:
    print("Skipping DiD plot: no DiD results available.")


NameError: name 'features_use' is not defined

## 9. Summary and Conclusions

In [12]:
print("=" * 60)
print("ANALYSIS SUMMARY")
print("=" * 60)

print(f"""
Dataset: Sade-Feldman et al. (Cell 2018) - Melanoma Immunotherapy
  - Total cells: {adata.n_obs:,}
  - Participants: {adata.obs['participant_id'].nunique()}
  - Response groups: Responder vs Non-responder (participant-level harmonized)
  - Time points: Pre-treatment, Post-treatment (on therapy)

Longitudinal pairing:
  - Paired participants: {n_paired}
  - Paired Responders: {paired_by_response.get('Responder', 0)}
  - Paired Non-responders: {paired_by_response.get('Non-responder', 0)}

Analysis approach:
  1. Cross-sectional: Responder vs Non-responder at each timepoint (OLS)
  2. Within-arm longitudinal: Pre→Post changes (Wilcoxon signed-rank)
  3. Difference-in-Differences: Differential changes (Mann-Whitney U on deltas)

Statistical methods:
  - Participant-level aggregation (avoids pseudoreplication)
  - FDR correction for multiple testing (alpha={FDR_ALPHA})
  - Non-parametric tests for robustness with small samples
""")

# Summary of key findings
print("")
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

# Cross-sectional
if not all_cross.empty:
    sig_cross = all_cross[all_cross["FDR_arm"] < FDR_ALPHA]
    if not sig_cross.empty:
        print("")
        print(f"Cross-sectional (FDR < {FDR_ALPHA}):")
        for _, row in sig_cross.iterrows():
            direction = "↑ Responder" if row["beta_arm"] > 0 else "↑ Non-responder"
            print(f"  {row['feature']} at {row['visit']}: {direction} (p={row['p_arm']:.3f})")
    else:
        print("")
        print(f"Cross-sectional: No significant differences (FDR < {FDR_ALPHA})")

# Within-arm
if not all_within.empty:
    sig_within = all_within[(all_within["FDR_time"].notna()) & (all_within["FDR_time"] < FDR_ALPHA)]
    if not sig_within.empty:
        print("")
        print(f"Within-arm changes (FDR < {FDR_ALPHA}):")
        for _, row in sig_within.iterrows():
            direction = "↑" if row["mean_delta"] > 0 else "↓"
            print(f"  {row['feature']} in {row['arm']}: {direction} (delta={row['mean_delta']:.3f})")
    else:
        print("")
        print(f"Within-arm: No significant changes (FDR < {FDR_ALPHA})")

# DiD
if did_results is not None and not did_results.empty:
    sig_did = did_results[(did_results["FDR_DiD"].notna()) & (did_results["FDR_DiD"] < FDR_ALPHA)]
    if not sig_did.empty:
        print("")
        print(f"Differential changes (DiD, FDR < {FDR_ALPHA}):")
        for _, row in sig_did.iterrows():
            direction = "Responders ↑ more" if row["beta_DiD"] > 0 else "Non-responders ↑ more"
            print(f"  {row['feature']}: {direction} (β={row['beta_DiD']:.3f})")
    else:
        print("")
        print(f"DiD: No significant differential changes (FDR < {FDR_ALPHA})")

print("")
print("=" * 60)


ANALYSIS SUMMARY


NameError: name 'adata' is not defined